# Integrated Rolling Simulations

This notebook integrates my 2.4/2.5/2.7 modules with teammate processed rolling data and fitted regression outputs. It does not modify teammate files.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    if start.name == "notebooks":
        start = start.parent
    for candidate in [start] + list(start.parents):
        if (candidate / "data").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find project root")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)

/Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project


In [2]:
from src.integration_config import TeammateDataConfig, IntegratedRunConfig
from src.teammate_integration import (
    load_rolling_pair_summary, parse_pair_universe, load_monthly_bin_data_for_pair,
    load_baseline_20stocks,
)
from src.fitted_regression_params import load_ow_transient_params, load_reduced_form_params
from src.integrated_alpha_runner import build_alpha_input_for_pair
from src.integrated_ow_runner import run_my_ow_strategy_on_pair
from src.reduced_form_regression_evaluator import evaluate_marginal_impact_from_strategy_trades
from src.integrated_reporting import save_pair_plots
from src.integrated_validation import validate_pair_inputs

## 1. Load Rolling Pair Summary and Params

In [3]:
data_config = TeammateDataConfig()
run_config = IntegratedRunConfig(run_mode="single_pair", debug_pair_id=1, save_pair_level_trades=True)
pairs = load_rolling_pair_summary(data_config, PROJECT_ROOT)
ow_params = load_ow_transient_params(PROJECT_ROOT / data_config.ow_params_path)
rf_params = load_reduced_form_params(PROJECT_ROOT / data_config.reduced_form_params_path)
display(pairs.head())
display(ow_params.head())
display(rf_params.head())

,pair_id,train_month,test_month,train_bin_path,test_bin_path,train_fill_path,test_fill_path,n_train_stocks,n_test_stocks,n_common_stocks_pair_universe,top20_common_stocks_by_train_notional
0,1,201901,201902,/Users/ulysse/Desktop/Studies/Imperial/Computi...,/Users/ulysse/Desktop/Studies/Imperial/Computi...,/Users/ulysse/Desktop/Studies/Imperial/Computi...,/Users/ulysse/Desktop/Studies/Imperial/Computi...,50,50,50,"['AMZN', 'AAPL', 'AMD', 'AMGN', 'ADBE', 'ALGN'..."
1,2,201902,201903,/Users/ulysse/Desktop/Studies/Imperial/Computi...,/Users/ulysse/Desktop/Studies/Imperial/Computi...,/Users/ulysse/Desktop/Studies/Imperial/Computi...,/Users/ulysse/Desktop/Studies/Imperial/Computi...,50,50,50,"['AMZN', 'AAPL', 'AMD', 'ADBE', 'AMGN', 'AMAT'..."
2,3,201903,201904,/Users/ulysse/Desktop/Studies/Imperial/Computi...,/Users/ulysse/Desktop/Studies/Imperial/Computi...,/Users/ulysse/Desktop/Studies/Imperial/Computi...,/Users/ulysse/Desktop/Studies/Imperial/Computi...,50,50,50,"['AMZN', 'AAPL', 'AMD', 'ADBE', 'AMGN', 'ALGN'..."
3,4,201904,201905,/Users/ulysse/Desktop/Studies/Imperial/Computi...,/Users/ulysse/Desktop/Studies/Imperial/Computi...,/Users/ulysse/Desktop/Studies/Imperial/Computi...,/Users/ulysse/Desktop/Studies/Imperial/Computi...,50,50,50,"['AMZN', 'AAPL', 'AMD', 'ADBE', 'AMGN', 'ANTM'..."
4,5,201905,201906,/Users/ulysse/Desktop/Studies/Imperial/Computi...,/Users/ulysse/Desktop/Studies/Imperial/Computi...,/Users/ulysse/Desktop/Studies/Imperial/Computi...,/Users/ulysse/Desktop/Studies/Imperial/Computi...,50,50,50,"['AMZN', 'AAPL', 'AMD', 'ADBE', 'AMGN', 'APC',..."


,pair_id,train_month,test_month,model,stock,half_life_sec,intercept,x_flow,ow_state_pre,x_trade,x_hidden,x_flow_depth,lobImb,effLobImb,spread_bps
0,1,201901,201902,OW_transient,A,1800.0,0.061743,167.390953,1.852045,NaN,NaN,NaN,NaN,NaN,NaN
1,1,201901,201902,OW_transient,AAL,60.0,-0.097961,962.578316,47.455393,NaN,NaN,NaN,NaN,NaN,NaN
2,1,201901,201902,OW_transient,AAP,60.0,0.041398,129.971500,23.288254,NaN,NaN,NaN,NaN,NaN,NaN
3,1,201901,201902,OW_transient,AAPL,30.0,0.025702,527.449573,-81.417252,NaN,NaN,NaN,NaN,NaN,NaN
4,1,201901,201902,OW_transient,ABBV,600.0,-0.007223,740.309541,-6.166051,NaN,NaN,NaN,NaN,NaN,NaN


,pair_id,train_month,test_month,model,stock,half_life_sec,intercept,x_flow,ow_state_pre,x_trade,x_hidden,x_flow_depth,lobImb,effLobImb,spread_bps
0,1,201901,201902,reduced_form,A,NaN,-0.061747,186.694998,NaN,-114.625991,-567.406033,-0.000532,0.442003,0.038193,0.057225
1,1,201901,201902,reduced_form,AAL,NaN,-0.158986,134.805581,NaN,462.487907,-358.235124,0.170464,0.618070,0.069178,0.039948
2,1,201901,201902,reduced_form,AAP,NaN,0.029139,91.304219,NaN,118.046071,-1135.576860,0.008796,0.480371,0.137565,0.004226
3,1,201901,201902,reduced_form,AAPL,NaN,0.032128,231.361169,NaN,133.472890,-261.445658,0.024833,0.207072,0.073076,0.018331
4,1,201901,201902,reduced_form,ABBV,NaN,0.001541,172.168259,NaN,458.017247,-202.952678,0.109357,0.407961,-0.071217,0.021520


## 2. Select Debug Pair and Load Train/Test Data

Requires `pyarrow` or `fastparquet` because teammate processed data are parquet files.

In [4]:
pair_row = pairs[pairs["pair_id"].eq(run_config.debug_pair_id)].iloc[0]
stocks = parse_pair_universe(pair_row)
print("Pair:", pair_row["pair_id"], pair_row["train_month"], "->", pair_row["test_month"])
print("Stocks:", stocks)
try:
    train_raw, test_raw = load_monthly_bin_data_for_pair(pair_row, stocks, data_config, sample_nrows=100000)
    display(train_raw.head())
    display(test_raw.head())
except ImportError as exc:
    print(exc)
    train_raw = pd.DataFrame()
    test_raw = pd.DataFrame()

Pair: 1 201901 -> 201902
Stocks: ['AMZN', 'AAPL', 'AMD', 'AMGN', 'ADBE', 'ALGN', 'AMAT', 'ANTM', 'AAL', 'ADP', 'ABBV', 'ADI', 'ABT', 'ADSK', 'ALXN', 'AGN', 'ACN', 'AAP', 'AMT', 'APD']


,date,time,stock,trade,orderFlow,hidden,auction,mid,midEnd,spread,depth,lobImb,effLobImb,datetime,trading_date,seconds_from_open,timestamp
0,2019-01-02,09:30:00,AAL,34.0,-1266.0,100.0,67717.0,31.525,31.525,0.085,494.4286,0.333333,0.660128,2019-01-02 09:30:00,2019-01-02,0.0,2019-01-02 09:30:00
1,2019-01-02,09:30:10,AAL,0.0,-200.0,0.0,0.0,31.520,31.520,0.050,443.0000,-0.097065,NaN,2019-01-02 09:30:10,2019-01-02,10.0,2019-01-02 09:30:10
2,2019-01-02,09:30:20,AAL,-8.0,2200.0,100.0,0.0,31.515,31.535,0.045,645.8571,0.333333,0.148389,2019-01-02 09:30:20,2019-01-02,20.0,2019-01-02 09:30:20
3,2019-01-02,09:30:30,AAL,-100.0,-417.0,100.0,0.0,31.515,31.515,0.025,551.1667,0.333333,0.000000,2019-01-02 09:30:30,2019-01-02,30.0,2019-01-02 09:30:30
4,2019-01-02,09:30:40,AAL,200.0,-2200.0,100.0,0.0,31.485,31.485,0.025,552.3333,0.846154,0.869159,2019-01-02 09:30:40,2019-01-02,40.0,2019-01-02 09:30:40


,date,time,stock,trade,orderFlow,hidden,auction,mid,midEnd,spread,depth,lobImb,effLobImb,datetime,trading_date,seconds_from_open,timestamp
0,2019-02-01,09:30:00,AAL,-899.0,907.0,1196.0,0.0,35.950,35.865,0.050,1208.6000,-0.482759,0.580433,2019-02-01 09:30:00,2019-02-01,0.0,2019-02-01 09:30:00
1,2019-02-01,09:30:10,AAL,272.0,1420.0,32.0,0.0,35.825,35.840,0.015,462.0000,0.470588,0.000000,2019-02-01 09:30:10,2019-02-01,10.0,2019-02-01 09:30:10
2,2019-02-01,09:30:20,AAL,200.0,5800.0,0.0,0.0,35.875,35.870,0.035,440.0000,0.333333,0.666667,2019-02-01 09:30:20,2019-02-01,20.0,2019-02-01 09:30:20
3,2019-02-01,09:30:30,AAL,410.0,5140.0,100.0,0.0,35.930,35.960,0.030,1020.0000,0.636364,0.777778,2019-02-01 09:30:30,2019-02-01,30.0,2019-02-01 09:30:30
4,2019-02-01,09:30:40,AAL,2323.0,14217.0,400.0,0.0,35.965,36.030,0.015,901.6667,0.714286,0.267606,2019-02-01 09:30:40,2019-02-01,40.0,2019-02-01 09:30:40


## 3. Validate Inputs

In [5]:
if len(train_raw) and len(test_raw):
    checks = validate_pair_inputs(pair_row, train_raw, test_raw, ow_params)
    display(pd.DataFrame(checks))

,check,status,message
0,train_nonempty,PASS,rows=56595
1,test_nonempty,PASS,rows=59129
2,required_columns,PASS,train_missing=[] test_missing=[]
3,stock_overlap,PASS,n_overlap=2
4,params_available,PASS,missing_param_stocks=[]


## 4. Generate Alpha on Test Month

In [6]:
if len(test_raw):
    pair_dir = PROJECT_ROOT / "outputs" / "rolling_runs" / f"pair_{int(pair_row['pair_id'])}"
    alpha = build_alpha_input_for_pair(
        test_raw, int(pair_row["pair_id"]), str(pair_row["test_month"]), stocks,
        output_dir=pair_dir / "alpha",
    )
    display(alpha[["stock", "timestamp", "mid", "future_return_h", "alpha_raw", "alpha_for_strategy"]].head())

/Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project/src/alpha.py:155: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  out["row_id_within_stock_date"] = out.groupby(
/Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project/src/alpha.py:159: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  out.groupby([config.stock_col, config.date_col], sort=False)[config.timestamp_col]
/Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project/src/alpha.py:191: FutureWarning: The default of observed=False is depreca

Time-gap diagnostics: median=10.00s, p95=10.00s, max=90.00s


,stock,timestamp,mid,future_return_h,alpha_raw,alpha_for_strategy
0,AAL,2019-02-01 09:30:00,35.950,0.006259,0.000121,0.000121
1,AAL,2019-02-01 09:30:10,35.825,0.009491,-0.000106,0.000116
2,AAL,2019-02-01 09:30:20,35.875,0.009756,0.000242,0.000119
3,AAL,2019-02-01 09:30:30,35.930,0.007515,0.000256,0.000122
4,AAL,2019-02-01 09:30:40,35.965,0.006117,-0.000313,0.000112


## 5. Run My OW Strategy

In [7]:
if len(test_raw):
    trades = run_my_ow_strategy_on_pair(pair_row, train_raw, test_raw, stocks, alpha, ow_params, run_config)
    display(trades.head())
    print("Total net pnl:", trades["net_pnl"].sum())

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

## 6. Evaluate Under Teammate Regressions

In [ ]:
if len(test_raw):
    ow_eval = evaluate_marginal_impact_from_strategy_trades(train_raw, test_raw, trades, ow_params, int(pair_row["pair_id"]), model_name="OW_transient")
    rf_eval = evaluate_marginal_impact_from_strategy_trades(train_raw, test_raw, trades, rf_params, int(pair_row["pair_id"]), model_name="reduced_form")
    display(ow_eval.head())
    display(rf_eval.head())
    print("OW eval pnl:", ow_eval["net_pnl_fitted_model"].sum())
    print("Reduced form eval pnl:", rf_eval["net_pnl_fitted_model"].sum())

## 7. Wrong Model Comparison

In [ ]:
if len(test_raw):
    comp = ow_eval[["stock", "datetime", "marginal_impact_bps"]].merge(
        rf_eval[["stock", "datetime", "marginal_impact_bps"]],
        on=["stock", "datetime"], suffixes=("_ow", "_rf")
    )
    comp["diff"] = comp["marginal_impact_bps_rf"] - comp["marginal_impact_bps_ow"]
    display(comp.describe())

## 8. Plots and Reports

In [ ]:
if len(test_raw):
    pair_dir = PROJECT_ROOT / "outputs" / "rolling_runs" / f"pair_{int(pair_row['pair_id'])}"
    save_pair_plots(pair_dir, trades, ow_eval, rf_eval)
    for p in sorted((pair_dir / "figures").glob("*.png")):
        print(p.relative_to(PROJECT_ROOT))

## 9. CLI Run

Run the default single-pair integration with:

```bash
python -m src.run_integrated_rolling_simulations --mode single_pair --pair-id 1 --save-trades
```

If parquet support is missing, install `pyarrow` or ask teammate for CSV exports.

## Deep Integrated Validation Report

This section reads the generated integrated report, validation checks, and pair-level debug table. It is designed to explain whether gross alpha capture, fitted impact costs, and PnL timing are behaving as expected.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

ROLLING_OUT = PROJECT_ROOT / 'outputs' / 'rolling_runs'
PAIR_ID = int(pair_row['pair_id']) if 'pair_row' in globals() else 1
PAIR_DIR = ROLLING_OUT / f'pair_{PAIR_ID}'

report_path = ROLLING_OUT / 'integrated_rolling_report.md'
validation_path = ROLLING_OUT / 'integrated_validation_checks.csv'
debug_path = PAIR_DIR / 'debug' / 'debug_strategy_path_sample.csv'

if report_path.exists():
    display(Markdown(report_path.read_text()))
else:
    print(f'Missing report: {report_path}')

## PnL / Wealth Formula Validation

In [ ]:
if validation_path.exists():
    validation_checks = pd.read_csv(validation_path)
    display(validation_checks)
    display(validation_checks.groupby('status').size().rename('n_checks').reset_index())
else:
    print(f'Missing validation checks: {validation_path}')

## Alpha Capture and Sizing Diagnostics

In [ ]:
trades_path = PAIR_DIR / 'my_ow_trades.csv'
if trades_path.exists():
    trades_debug = pd.read_csv(trades_path, parse_dates=['timestamp'])
    cols = ['alpha', 'future_return_h', 'position_before', 'position_after', 'signed_volume', 'gross_pnl', 'participation_rate']
    display(trades_debug[[c for c in cols if c in trades_debug.columns]].describe())
    print('corr(alpha, future_return_h):', trades_debug['alpha'].corr(trades_debug['future_return_h']) if 'future_return_h' in trades_debug else None)
    print('corr(position_before, future_return_h):', trades_debug['position_before'].corr(trades_debug['future_return_h']) if 'future_return_h' in trades_debug else None)
    print('gross_pnl:', trades_debug['gross_pnl'].sum())
    print('max participation:', trades_debug['participation_rate'].max())
else:
    print(f'Missing trades: {trades_path}')

## Fitted Regression Cost Sign Diagnostics

In [ ]:
for name in ['ow_transient_regression_evaluator.csv', 'reduced_form_regression_evaluator.csv']:
    p = PAIR_DIR / name
    if p.exists():
        ev = pd.read_csv(p, parse_dates=['datetime'])
        print('\n', name)
        print('total signed fitted cost:', ev['fitted_impact_cost_signed'].sum())
        print('total abs fitted cost:', ev['fitted_impact_cost_abs'].sum())
        print('net pnl fitted model:', ev['net_pnl_fitted_model'].sum())
        print('share positive cost:', (ev.loc[ev['signed_volume'].abs() > 0, 'fitted_impact_cost_signed'] > 0).mean())
        display(ev[['signed_volume','marginal_impact_bps','marginal_impact_price','fitted_impact_cost_signed','gross_pnl','net_pnl_fitted_model']].describe())
    else:
        print(f'Missing evaluator: {p}')

## One Stock/Day Path Inspection

In [ ]:
if debug_path.exists():
    debug_sample = pd.read_csv(debug_path, parse_dates=['timestamp'])
    display(debug_sample.head(30))
else:
    print(f'Missing debug sample: {debug_path}')

## Internal OW Wealth vs Fitted Evaluator Wealth

In [ ]:
from IPython.display import Image
figs = [
    PAIR_DIR / 'figures' / 'cumulative_wealth_internal_vs_fitted.png',
    PAIR_DIR / 'figures' / 'gross_pnl_vs_fitted_cost_cumulative.png',
    PAIR_DIR / 'figures' / 'marginal_impact_bps_histogram.png',
    PAIR_DIR / 'figures' / 'alpha_position_alignment_sample.png',
]
for fig in figs:
    if fig.exists():
        display(Image(filename=str(fig)))
    else:
        print(f'Missing figure: {fig}')

## Interpretation

Positive internal gross PnL indicates alpha capture before costs. If fitted-regression net PnL is negative while gross PnL is positive, the fitted model is estimating that our own order flow moves prices enough to offset the alpha capture. The validation checks above verify that this is not caused by a sign error in `orderFlow_scenario`, marginal impact bps, or the cost subtraction formula.

## Run Metadata and Figure Freshness

In [ ]:
import json
metadata_path = PROJECT_ROOT / 'outputs' / 'rolling_runs' / 'latest_run_metadata.json'
if metadata_path.exists():
    run_metadata = json.loads(metadata_path.read_text())
    display(run_metadata)
else:
    print(f'Missing metadata: {metadata_path}')

fig_checks = validation_checks[validation_checks['check'].astype(str).str.startswith('figure_')] if 'validation_checks' in globals() else pd.DataFrame()
if len(fig_checks):
    display(fig_checks)
else:
    print('Run the validation checks cell first, or rerun the integrated pipeline.')

## Fitted-Regression Proxy Strategy

The teammate `reduced_form` model is a regression for within-bin `ret_bps`, not a structural impact SDE. This section loads the local myopic quadratic-cost proxy strategy induced by the fitted regression's marginal impact slope. It should be interpreted as a fitted-model-aware proxy benchmark, not as a closed-form dynamic optimal AFS strategy.

In [ ]:
proxy_summary_path = PROJECT_ROOT / 'outputs' / 'rolling_runs' / 'all_pairs_fitted_proxy_summary.csv'
if proxy_summary_path.exists():
    fitted_proxy_summary = pd.read_csv(proxy_summary_path)
    display(fitted_proxy_summary)
else:
    print(f'Missing fitted proxy summary: {proxy_summary_path}')

In [ ]:
pair_id_for_proxy = pair_id if 'pair_id' in globals() else 1
proxy_trades_path = PROJECT_ROOT / 'outputs' / 'rolling_runs' / f'pair_{pair_id_for_proxy}' / 'fitted_proxy_strategy_trades.csv'
if proxy_trades_path.exists():
    fitted_proxy_trades = pd.read_csv(proxy_trades_path)
    display(fitted_proxy_trades[[
        'pair_id', 'stock', 'trading_date', 'datetime', 'mid', 'alpha', 'alpha_price',
        'impact_slope_bps_per_share', 'impact_slope_price_per_share', 'raw_trade',
        'signed_volume', 'position_before', 'position_after', 'participation_rate',
        'gross_pnl', 'fitted_impact_cost', 'net_pnl', 'cumulative_wealth'
    ]].head())
else:
    print(f'Missing fitted proxy trades: {proxy_trades_path}')

In [ ]:
from IPython.display import Image, display
fig_dir = PROJECT_ROOT / 'outputs' / 'rolling_runs' / f'pair_{pair_id_for_proxy}' / 'figures'
for name in [
    'fitted_proxy_cumulative_wealth.png',
    'ow_vs_fitted_proxy_wealth.png',
    'fitted_proxy_trade_histogram.png',
    'fitted_proxy_impact_slope_histogram.png',
    'fitted_proxy_participation_rate_histogram.png',
]:
    fig_path = fig_dir / name
    if fig_path.exists():
        display(Image(filename=str(fig_path)))
    else:
        print(f'Missing figure: {fig_path}')

## Forced Liquidation Stress Modes

Compare the hard-block liquidation stress against the cap-respecting residual liquidation stress. Hard block answers the project question directly; capped residual is the operational version with liquidation priority over alpha trades.

In [ ]:
stress_dir = PROJECT_ROOT / 'outputs' / 'rolling_runs' / f'pair_{pair_id_for_proxy}' / 'stress'
forced_summary_path = stress_dir / 'forced_liquidation_summary.csv'
if forced_summary_path.exists():
    forced_liq_summary = pd.read_csv(forced_summary_path)
    display(forced_liq_summary[[c for c in [
        'scenario_name', 'liquidation_mode', 'net_pnl_under_ow_regression_eval',
        'net_pnl_under_reduced_form_eval', 'number_of_liquidation_events',
        'number_fully_liquidated_immediately', 'number_fully_liquidated_eventually',
        'number_with_residual_after_first_liquidation', 'number_with_overnight_residual',
        'max_residual_position', 'total_alpha_trades_suppressed_due_to_liquidation',
        'hard_block_cap_violation_rate'
    ] if c in forced_liq_summary.columns]])
else:
    print(f'Missing forced liquidation summary: {forced_summary_path}')

In [ ]:
for name in [
    'forced_liq_hard_vs_capped_wealth.png',
    'forced_liq_event_costs_by_stock.png',
    'forced_liq_residual_inventory_by_stock.png',
    'forced_liq_time_to_liquidate.png',
    'forced_liq_participation_rates.png',
    'forced_liq_sample_path.png',
]:
    fig_path = fig_dir / name
    if fig_path.exists():
        display(Image(filename=str(fig_path)))
    else:
        print(f'Missing figure: {fig_path}')